<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/15_ekf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extended Kalman Filter

In this problem, you will implement an extended Kalman Filter to estimate the state of a unicycle model system.

Consider a unicycle model

$$ \dot{\mathbf{x}} = \begin{bmatrix} \dot{x} \\ \dot{y} \\ \dot{\theta} \end{bmatrix} = \begin{bmatrix} v\cos\theta \\ v\sin\theta \\ \omega \end{bmatrix}, \qquad \mathbf{u} = \begin{bmatrix} v \\ \omega \end{bmatrix} $$

Let the *discrete time* dynamics with Gaussian noise be given by the following.

$$ \mathbf{x}_{t+1} = f_d(\mathbf{x}_t, \mathbf{u}_t) + \mathbf{w}_t, \quad \mathbf{w}_t \sim \mathcal{N}(0, Q) $$

where $\mathbb{E}[\mathbf{w}_t\mathbf{w}_t^T] = Q$.

Assume that we only obtain GPS position measurements of the robot. The measurement model is

$$ \mathbf{y}_t = \begin{bmatrix} x_t\\ y_t \end{bmatrix} + \mathbf{v}_t $$

where $\mathbb{E}[\mathbf{v}_t\mathbf{v}_t^T] = R$.

In this problem, let $\Delta t = 0.1$, and to start off, let $Q = \mathrm{diag}([0.05, 0.05, 0.01])$, and $R=\mathrm{diag}([2.0, 2.0])$.

Suppose that at the beginning, our estimate of the initial state is $\mathbf{x}_0 \sim \mathcal{N}(\mu_0, \Sigma_0)$ where $\mu_0 = \begin{bmatrix} -4.0\\ 4.0\\ \frac{\pi}{4} + 0.5 \end{bmatrix}$, $\Sigma_0 = \mathrm{diag}([8., 8., 2.])$.
During the episode, the robot is executing control inputs according to the control law:

$$\mathbf{u}_k = \begin{bmatrix} 0.5\sin(0.5t_k) + 1 \\ \sin(t_k) \end{bmatrix}.$$

Let the *true* initial state of the system be $\mathbf{x}_0^\mathrm{true} = \begin{bmatrix} 0.0\\ 0.0\\ \frac{\pi}{4} \end{bmatrix}$.
Note that in "real life" we don't the know the exact ground truth value of the robot's state, but in this problem, it is given so that you can compare your estimate and generate noisy measurements.



In the following code cell, you will be required to implement an **Extended Kalman Filter**, and simulate the system (both the ground truth trajectory and the estimated trajectory).

In [ ]:
!pip install dynamaxsys==0.0.7

In [ ]:
# in this problem, we will use the dynamaxsys library to import dynamical systems implemented in JAX: https://github.com/UW-CTRL/dynamaxsys
from dynamaxsys import Dynamics, Unicycle
from dynamaxsys import get_discrete_time_dynamics
from dynamaxsys import linearize

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.patches import Ellipse
from scipy.stats import chi2
from typing import Callable


In [ ]:
# some helper functions. No need to change these functions


def wrap_to_pi(a: float) -> float:
    """
    Wrap angle to [-pi, pi].
    """
    return (a + np.pi) % (2 * np.pi) - np.pi


def plot_uncertainty_ellipse(
    ax: plt.Axes,
    mean: np.ndarray,
    cov: np.ndarray,
    confidence: float = 0.95,
    dim: int = 2,
    **kwargs,
):
    """
    Plot an uncertainty ellipse based on the covariance matrix.
    """
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    order = eigenvalues.argsort()[::-1]
    eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]
    angle = np.degrees(np.arctan2(*eigenvectors[:, 0][::-1]))
    chi2_val = chi2.ppf(confidence, df=dim)
    width, height = 2 * np.sqrt(chi2_val * eigenvalues)
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)

### (a) Implement the EKF step

In [ ]:
# TODO: complete the ekf function
# This is the EKF step.
# You will need to implement the EKF step in the function below.
# The EKF step consists of two main parts: the prediction step and the update step.
# In the prediction step, we use the robot dynamics to predict the next state and its covariance.
# In the update step, we use the measurement to update the predicted state and covariance.


def ekf_step(
    state_mean: jnp.ndarray,
    P: jnp.ndarray,
    control: jnp.ndarray,
    measurement: jnp.ndarray,
    Q: jnp.ndarray,
    R: jnp.ndarray,
    dt_robot_dynamics: Dynamics,
    measurement_model: Callable,
    time: float = 0.0,
):
    """
    Perform one step of the EKF algorithm.
    Inputs:
        state_mean: current state estimate
        P: current covariance estimate
        control: current control input
        measurement: measurement for next time step
        Q: process noise covariance
        R: measurement noise covariance
        dt_robot_dynamics: discrete-time robot dynamics function
        measurement_model: measurement model function
    Outputs:
        x_upd: updated state estimate at time t
        P_upd: updated covariance estimate at time t
    """
    # Prediction step
    # predict the next state given known robot dynamics
    # x_pred = ... # update me!

    # predict the next covariance given known robot dynamics
    # First, you need to linearize the dynamics and obtain the A matrix, linear term for state.
    # A = ... # update me!
    # P_pred = ... # update me!

    # Update step
    # compute innovation (measurement residual)
    # h = ... # update me!

    # Compute C matrix
    # linearize the measurement model an obtain C matrix, linear term for state
    # C = ... # update me!

    # Compute Kalman gain
    # K = ... # update me!

    # Update the state estimate and covariance
    # updated state estimate
    x_upd = ...  # update me!
    # updated covariance estimate
    P_upd = ... # update me!

    return x_upd, P_upd


### (b) Set up the rest of the problem parameters and functions

In [ ]:
# set up robot dynamics. Run cell as is.
# we can use the dynamaxsys library to import the robot dynamics
# the robot dynamics is a unicycle model
dt = 0.1
ct_robot_dynamics = Unicycle()  # robot dynamics
dt_robot_dynamics = get_discrete_time_dynamics(
    ct_robot_dynamics, dt=dt
)  # get discrete time dynamics
state_dim = dt_robot_dynamics.state_dim

#### Set up the robot control function

In [ ]:
# Run cell as is.
def u_func(t):
    """
    Control input function.
    This function generates a control input based on the time t.
    u = [v, omega]
    """
    return jnp.array([0.5 * jnp.sin(t) + 1,  jnp.sin(t)])


### Define the robot measurement model (without noise)

In [ ]:
# Run cell as is.
# this function obtains the measurement, in this case, GPS coordinates.
# The noise will be added later in the implement.
# Do not add noise in this function.

obs_dim = 2 # dimension of the observation
def measurement_model(state: jnp.ndarray, control: jnp.ndarray, time: float) -> jnp.ndarray:
    '''Implements the measurement model without the noise.
    The noise will be added in the EKF step.'''
    x, y, th = state
    return jnp.array([x, y])


### Define problem matrices and other variables
Try out different values and see how they affect the EKF performance

In [ ]:
# TODO: update these numbers are appropriate.
P0 = jnp.eye(state_dim)  # initial state estimate covariance
Q = jnp.eye(state_dim)  # process noise covariance
R = jnp.eye(obs_dim)  # measurement noise covariance
initial_state = jnp.zeros([state_dim])  # true initial state


### (c) Simulate the episode!

In [ ]:
# Run cell as is. Also feel free to change the setup parameters as you wish.
n_timesteps = 50  # number of timesteps of run

# set up lists to store the state estimates, true states, covariances, measurements, and times
# set initial state estimate with some error
xs_est = [initial_state + 0.1 * jnp.array([-4.0, 4.0, 0.5])]  # mean initial state estimate
xs_true = [initial_state] # keep track of true states
Ps = [P0]  # initial covariance
measurements = []
ts = []

# seed the random number generator
# and sample the process noise and measurement noise
key = jax.random.PRNGKey(0)
dyn_noise = jax.random.multivariate_normal(
    key, jnp.zeros(state_dim), Q, shape=(n_timesteps,)
)  # sample from the process noise
measurement_noise = jax.random.multivariate_normal(
    key, jnp.zeros(obs_dim), R, shape=(n_timesteps,)
)  # sample from the measurement noise


# run the EKF algorithm over multiple timesteps
for ti in range(n_timesteps):
    t = ti * dt # get time
    ts.append(t)

    # get the control input
    u = u_func(t)

    # get the true state with process noise
    x_next_true = dt_robot_dynamics(xs_true[-1], u, t) + dyn_noise[ti] # add noise to the true state

    # wrap the angle to [-pi, pi]
    x_next_true = x_next_true.at[2].set(wrap_to_pi(x_next_true[2])) # wrap the angle to [-pi, pi]

    # get the measurement with measurement noise
    z = measurement_model(x_next_true, u, t) + measurement_noise[ti] # add noise to the measurement

    # wrap the angle to [-pi, pi]
    z = z.at[2].set(wrap_to_pi(z[2])) # wrap the angle to [-pi, pi]

    # perform one step of the EKF algorithm
    x, P = ekf_step(xs_est[-1], Ps[-1], u, z, Q, R, dt_robot_dynamics, measurement_model) # perform one step of the EKF algorithm

    # wrap the angle to [-pi, pi]
    x = x.at[2].set(wrap_to_pi(x[2])) # wrap the angle to [-pi, pi]

    # add the new state estimate, true state, covariance, and measurement to the lists
    xs_est.append(x)
    xs_true.append(x_next_true)
    measurements.append(z)
    Ps.append(P)


ts.append(n_timesteps * dt)
xs_est = jnp.stack(xs_est)
xs_true = jnp.stack(xs_true)
measurements = jnp.stack(measurements)
Ps = jnp.stack(Ps)
ts = jnp.array(ts)

Plot your results below!
Uncomment the plotting code below to see your results!

In [ ]:
# Run cell as is.
confidence = 0.95
scale = jnp.sqrt(chi2.ppf(0.95, df=3))

plt.figure(figsize=(9, 8))
plt.subplot(2,2,1)
ax = plt.gca()
plt.plot(xs_true[:, 0], xs_true[:, 1], 'o-', label='True trajectory', color='blue', markersize=3)
plt.plot(xs_est[:, 0], xs_est[:, 1], 'o-', label='Estimated trajectory', color='red', markersize=3)
plt.scatter(xs_true[:, 0], xs_true[:, 1], color='blue', s=10)
plt.scatter(xs_est[:, 0], xs_est[:, 1], color='red', s=10)
plt.scatter(measurements[:, 0], measurements[:, 1], color='green', s=10, label='Measurements')
plt.scatter(xs_est[0, 0], xs_est[0, 1], color='black', s=30, label='Initial state')
plt.scatter(xs_true[0, 0], xs_true[0, 1], color='black', s=30)

plot_uncertainty_ellipse(ax, xs_est[0][:2], Ps[0][:2,:2], confidence=confidence, alpha=0.1, label="Uncertainty 95%")
for (mu, sigma) in zip(xs_est[1:], Ps[1:]):
    plot_uncertainty_ellipse(ax, mu[:2], sigma[:2,:2], confidence=confidence, alpha=0.1)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Trajectory')
plt.legend()
plt.axis('equal')
plt.grid()

plt.subplot(2,2,2)
plt.plot(ts, xs_true[:, 0], label='True x position', color='blue')
plt.plot(ts, xs_est[:, 0], label='Estimated x position', color='red')
plt.scatter(ts[1:], measurements[:, 0], color='green', s=10, label='Measurements')
plt.errorbar(ts, xs_est[:, 0], yerr=scale * jnp.sqrt(Ps[:, 0, 0]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('x position (m)')
plt.title('x position')
plt.legend()
plt.grid()

plt.subplot(2,2,3)
plt.plot(ts, xs_true[:, 1], label='True y position', color='blue')
plt.plot(ts, xs_est[:, 1], label='Estimated y position', color='red')
plt.scatter(ts[1:], measurements[:, 1], color='green', s=10, label='Measurements')
plt.errorbar(ts, xs_est[:, 1], yerr=scale * jnp.sqrt(Ps[:, 1, 1]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('y position (m)')
plt.title('y position')
plt.legend()
plt.grid()

plt.subplot(2,2,4)
plt.plot(ts, xs_true[:, 2], label='True angle', color='blue')
plt.plot(ts, xs_est[:, 2], label='Estimated angle', color='red')
plt.errorbar(ts, xs_est[:, 2], yerr=scale * jnp.sqrt(Ps[:, 2, 2]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('Angle (rad)')
plt.title('heading')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

## (d) EKF-SLAM
The EKF algorithm is not only useful for estimating the state of the robot, but it can also be used to estimate the location of landmarks in the environment. That is, we can use the EKF algorithm estimate both the robot state (i.e., localization) and also estimate a map of its surrounding (i.e., mapping). This is referred to as SLAM--Simultaneous Localization And Mapping.

In this problem we will implement a *very simple* SLAM problem by using the above code, but we add one new landmark to the problem whose position we wish to estimate.

The core idea is to augment the state vector with the location of the landmark. Essentially treat the landmark location as part of the state, update the measurement model and other matrices accordingly, and then apply the same old EKF algorithm on this augmented problem.

(Note: This is a nice example where the algorithm we use is relatively straightforward, but by carefully formulating the problem a certain way, these "simple" algorithms can be very powerful!)

In addition to the GPS measurements, the robot receives noisy range and bearing measurements from a fixed landmark.
Let the landmark position be $\ell = \begin{bmatrix} \ell_x \\ \ell_y \end{bmatrix}$. Then it receivees a noisy measurement for the landmark's position:

$$ \tilde{\mathbf{y}}_t = \begin{bmatrix} r \\ \phi \end{bmatrix} = \begin{bmatrix} \sqrt{(\ell_x - x)^2 + (\ell_y - y)^2} \\ \arctan2(\ell_y - y, \ell_x - x) - \theta \end{bmatrix} + \tilde{\mathbf{v}}_t, \quad \tilde{\mathbf{v}}_t \sim \mathcal{N}(0,\tilde{R}) $$

Note: The above is referring to the `arctan2` function which considers the quadrant when computing the angle.

We augment the state vector to be $\begin{bmatrix} x \\ y \\ \theta \\ \ell_x \\ \ell_y \end{bmatrix}$. To spice things up a little, let's assume the landmarks move according to the continuous-time dynamics,

$$ \begin{bmatrix} \dot{\ell}_x \\ \dot{\ell}_y \end{bmatrix} = \begin{bmatrix} -\ell_y \\ 0.1\ell_x \end{bmatrix}$$

#### (d)(i) Define the dynamics for the augmented state

below is the augmented unicycle dynamics with the landmark states and dynamics included.

In [ ]:
# Run cell as is.
from dynamaxsys.base import ControlAffineDynamics


# define the continuous time dynamics of the unicycle with landmarks
class UnicycleLandmark(ControlAffineDynamics):
    """
    Unicycle dynamics with landmarks.
    The state is [x, y, theta, lx, ly], where (lx, ly) are the coordinates of the landmark.
    The control input is [v, omega], where v is the linear velocity and omega is the angular velocity.

    The dynamics are given by:
    dx/dt = v * cos(theta)
    dy/dt = v * sin(theta)
    dtheta/dt = omega
    dlx/dt = -ly
    dly/dt = 0.1 * lx

    The dynamics and control affine, and has the following form:
    dx/dt = f(x) + g(x) * u
    where f(x) is the drift dynamics and g(x) is the control Jacobian.
    """

    state_dim: int = 5
    control_dim: int = 2

    def __init__(self):
        def drift_dynamics(state, time):
            _, _, _, lx, ly = state
            return jnp.array([0., 0., 0., -ly, 0.1*lx])

        def control_jacobian(state, time):
            _, _, th, _, _ = state
            # v, om = control
            return jnp.array(
                [
                    [jnp.cos(th), 0.],
                    [jnp.sin(th), 0.],
                    [0., 1.],
                    [0., 0.],
                    [0., 0.]
                ]
            )

        super().__init__(drift_dynamics, control_jacobian, self.state_dim, self.control_dim)



# compute the discrete time dynamics of the unicycle with landmarks given the continuous time dynamics
ct_robot_dynamics = UnicycleLandmark()  # robot dynamics
dt = 0.1
dt_robot_dynamics = get_discrete_time_dynamics(
    ct_robot_dynamics, dt=dt
)  # discrete time dynamics
state_dim = dt_robot_dynamics.state_dim
control_dim = dt_robot_dynamics.control_dim


#### (d)(ii) Define the measurement model (without noise)

In [ ]:
# TODO: update the obsevation model to include landmark observations.
obs_dim = 5
def measurement_landmark_model(state, control, time):
    x, y, th, lx, ly = state
    return  jnp.array([x,
                       y,
                       th,
                       ..., # update me
                       ...] # update me
    )


#### (d)(iii) Initialize the problem matrices and variables

We will use the same noise covariances and initial states as before.
But for the landmark, suppose the process noise covariance for the landmark is $\tilde{Q} = \mathrm{diag}([0.1, 0.1])$, and the measurement noise covariance is $\tilde{R} = \mathrm{diag}([4., 4.])$

Let the true initial position of the landmark be $\begin{bmatrix} \ell_x^\mathrm{true}\\ \ell_y^\mathrm{true}\end{bmatrix} = \begin{bmatrix} 5.0 \\ 5.0 \end{bmatrix}$, and the initial estimate be $\begin{bmatrix} \ell_x^\mathrm{est}\\ \ell_y^\mathrm{est}\end{bmatrix} \sim \mathcal{N}(\begin{bmatrix} 1.0 \\ -1.0 \end{bmatrix}, \mathrm{diag}([5., 5.]))$



In [ ]:
# feel free to change these numbers as appropriate
P0 = jnp.diag(jnp.array([8., 8., 2., 5., 5.])) # initial covariance
Q = jnp.diag(jnp.array([0.1, 0.1, 0.01, 0.1, 0.1]))
R = jnp.diag(jnp.array([5., 5., .1, 4., 4.]))
x0 = jnp.array([0.0, 0.0, jnp.pi/4, 5., 5.]) # true initial state


#### (d)(iv) Simulate the episode!

In [ ]:
# Run cell as is. Also feel free to change the setup parameters as you wish.
n_timesteps = 150  # number of timesteps of run

# set up lists to store the state estimates, true states, covariances, measurements, and times
xs_est = [x0 + jnp.array([-4.0, 4.0, 0.5, 1.0, -1.0])]  # initial state
xs_true = [x0]
Ps = [P0]  # initial covariance
measurements = []
ts = []

# seed the random number generator
# and sample the process noise and measurement noise
key = jax.random.PRNGKey(0)
dyn_noise = jax.random.multivariate_normal(
    key, jnp.zeros(state_dim), Q, shape=(n_timesteps,)
)  # sample from the process noise
measurement_noise = jax.random.multivariate_normal(
    key, jnp.zeros(obs_dim), R, shape=(n_timesteps,)
)  # sample from the measurement noise


# run the EKF algorithm over multiple timesteps

for ti in range(n_timesteps):
    t = ti * dt # get time
    ts.append(t)

    # get the control input
    u = u_func(t)

    # get the true state with process noise
    x_next_true = dt_robot_dynamics(xs_true[-1], u, t) + dyn_noise[ti] # add noise to the true state

    # wrap the angle to [-pi, pi]
    x_next_true = x_next_true.at[2].set(wrap_to_pi(x_next_true[2])) # wrap the angle to [-pi, pi]

    # get the measurement with measurement noise
    z = measurement_landmark_model(x_next_true, u, t) + measurement_noise[ti] # add noise to the measurement

    # wrap the angle to [-pi, pi]
    z = z.at[2].set(wrap_to_pi(z[2])) # wrap the angle to [-pi, pi]

    # perform one step of the EKF algorithm
    x, P = ekf_step(xs_est[-1], Ps[-1], u, z, Q, R, dt_robot_dynamics, measurement_landmark_model) # perform one step of the EKF algorithm

    # wrap the angle to [-pi, pi]
    x = x.at[2].set(wrap_to_pi(x[2])) # wrap the angle to [-pi, pi]

    xs_est.append(x)
    xs_true.append(x_next_true)
    measurements.append(z)
    Ps.append(P)


# TODO: uncomment the lines below

ts.append(n_timesteps * dt)
xs_est = jnp.stack(xs_est)
xs_true = jnp.stack(xs_true)
measurements = jnp.stack(measurements)
Ps = jnp.stack(Ps)
ts = jnp.array(ts)

Plot your results below!
Uncomment the plotting code below to see your results!

In [ ]:
# Run cell as is.

confidence = 0.95
scale = jnp.sqrt(chi2.ppf(0.95, df=5))

plt.figure(figsize=(12, 8))
plt.subplot(2,2,1)
ax = plt.gca()
plt.plot(xs_true[:, 0], xs_true[:, 1], 'o-', label='True trajectory', color='blue', markersize=3)
plt.plot(xs_est[:, 0], xs_est[:, 1], 'o-', label='Estimated trajectory', color='red', markersize=3)
plt.scatter(xs_true[:, 0], xs_true[:, 1], color='blue', s=10)
plt.scatter(xs_est[:, 0], xs_est[:, 1], color='red', s=10)
plt.scatter(measurements[:, 0], measurements[:, 1], color='green', s=10, label='Measurements')
plt.scatter(xs_est[0, 0], xs_est[0, 1], color='black', s=30, label='Initial state')
plt.scatter(xs_true[0, 0], xs_true[0, 1], color='black', s=30)

plot_uncertainty_ellipse(ax, xs_est[0][:2], Ps[0][:2,:2], confidence=0.95, alpha=0.1, label="Uncertainty 95%")
for (mu, sigma) in zip(xs_est[1:], Ps[1:]):
    plot_uncertainty_ellipse(ax, mu[:2], sigma[:2,:2], confidence=0.95, alpha=0.1)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Trajectory')
plt.legend(ncol=2)
plt.axis('equal')
plt.grid()

plt.subplot(2,2,2)
plt.plot(ts, xs_true[:, 0], label='True x position', color='blue')
plt.plot(ts, xs_est[:, 0], label='Estimated x position', color='red')
plt.scatter(ts[1:], measurements[:, 0], color='green', s=10, label='Measurements')
plt.errorbar(ts, xs_est[:, 0], yerr=scale * jnp.sqrt(Ps[:, 0, 0]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('x position (m)')
plt.title('x position')
plt.legend()
plt.grid()

plt.subplot(2,2,3)
plt.plot(ts, xs_true[:, 1], label='True y position', color='blue')
plt.plot(ts, xs_est[:, 1], label='Estimated y position', color='red')
plt.scatter(ts[1:], measurements[:, 1], color='green', s=10, label='Measurements')
plt.errorbar(ts, xs_est[:, 1], yerr=scale * jnp.sqrt(Ps[:, 1, 1]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('y position (m)')
plt.title('y position')
plt.legend()
plt.grid()

plt.subplot(2,2,4)
plt.plot(ts, xs_true[:, 2], label='True angle', color='blue')
plt.plot(ts, xs_est[:, 2], label='Estimated angle', color='red')
plt.scatter(ts[1:], measurements[:, 2], color='green', s=10, label='Measurements')
plt.errorbar(ts, xs_est[:, 2], yerr=scale * jnp.sqrt(Ps[:, 2, 2]), fmt='.', color='red', alpha=0.2, label='Uncertainty 95%')
plt.xlabel('Time')
plt.ylabel('Angle (rad)')
plt.title('heading')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 3))
plt.subplot(1,3,1)
ax = plt.gca()
plt.scatter(xs_true[:, 3], xs_true[:, 4], label='True Landmark Location', color='blue', s=20, marker='o')
plt.scatter(xs_est[:, 3], xs_est[:, 4], label='Estimated Landmark Location', color='red', s=20, marker='o')
for (mu, sigma) in zip(xs_est, Ps):
    plot_uncertainty_ellipse(ax, mu[3:], sigma[3:,3:], confidence=0.95, alpha=0.1)
plt.xlabel('Landmark x position (m)')
plt.ylabel('Landmark y position (m)')
plt.title('Landmark Trajectory')
plt.legend()
plt.grid()
plt.axis('equal')

plt.subplot(1,3,2)
plt.plot(ts, xs_true[:, 3], label='True Landmark x position', color='blue')
plt.plot(ts, xs_est[:, 3], label='Estimated Landmark x position', color='red')
plt.errorbar(ts, xs_est[:, 3], yerr=jnp.sqrt(Ps[:, 3, 3]), fmt='.', color='red', alpha=0.2, label='Uncertainty')
plt.xlabel('Time')
plt.ylabel('Landmark x position (m)')
plt.title('Landmark x position')
plt.legend()
plt.grid()

plt.subplot(1,3,3)
plt.plot(ts, xs_true[:, 4], label='True Landmark y position', color='blue')
plt.plot(ts, xs_est[:, 4], label='Estimated Landmark y position', color='red')
plt.errorbar(ts, xs_est[:, 4], yerr=jnp.sqrt(Ps[:, 4, 4]), fmt='.', color='red', alpha=0.2, label='Uncertainty')
plt.xlabel('Time')
plt.ylabel('Landmark y position (m)')
plt.title('Landmark y position')
plt.legend()
plt.grid()
plt.tight_layout()
plt.subplots_adjust(wspace=0.4)
plt.suptitle('Landmark Position Estimation')
plt.subplots_adjust(top=0.85)
plt.show()

### (e) Interpret your results
This is an open-ended question. Do some exploration and see what kind of results you observed for different values values for $Q$ and $R$.
Some questions to consider:
- What happens if $Q$ is larger than $R$ and vice versa?
- What if the $Q$ and $R$ values you pick for the EKF does not match the true noise covariance of the true system? (i.e., the covariance matrix used to generate the noise differs from your choice of $Q$ and $R$)
- The heading is probably not handled in the best way in this problem (wrap_to_pi was applied but this causes a discontinous jump at times). Are there better ways to handle this?
- Does the choice of control inputs affect the estimation performance?